# 03 — Explainability with SHAP
**Objective 3 of the proposal:** apply SHAP to generate global and local explanations of model behaviour, identify the features that most strongly drive fraud predictions, and cross-check the results against the logistic regression baseline.

**Run `1_EDA.ipynb` and `2_Modelling.ipynb` first** — this notebook loads their saved models and test data. Nothing is retrained here.

**Method**
- LightGBM is explained with **TreeSHAP** (Lundberg et al., 2020), using LightGBM's built-in implementation (`pred_contrib=True`). It is exact and fast enough to explain *every* application in the test set, so no down-sampling is needed for the numbers (plots use a sample for readability).
- SHAP values are in **log-odds** units: positive values push an application towards "fraud", negative values towards "legitimate". For each application, the SHAP values plus the base value add up exactly to the model's score.
- Logistic regression is explained with the exact linear SHAP formula, SHAP = coefficient × (feature value − average feature value), so both models are compared on the same scale.

In [ ]:
import os
import sys
import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore', message='IProgress not found')   # harmless progress-bar notice
import shap
from scipy.stats import spearmanr

RANDOM_STATE = 42
N_PLOT = 10_000   # applications shown in beeswarm/dependence plots (all are used for the numbers)

# ---- Work out where the data is, depending on which machine this is running on ----
COLAB_DATA_DIR = '/content/drive/MyDrive/Dissertation/data'   # edit to match your Drive folder

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path(COLAB_DATA_DIR)
    MACHINE = 'Google Colab'
elif os.environ.get('DISSERTATION_DATA_DIR'):
    DATA_DIR = Path(os.environ['DISSERTATION_DATA_DIR'])
    MACHINE = 'Local (DISSERTATION_DATA_DIR)'
else:
    here = Path.cwd().resolve()
    DATA_DIR = next(
        (p / 'data' for p in [here, *here.parents]
         if (p / 'data' / 'Base.csv').exists() or (p / 'data' / 'Base.parquet').exists()),
        None,
    )
    if DATA_DIR is None:
        raise FileNotFoundError(
            "Could not find data/Base.csv. Put it in a 'data' folder inside the repository, "
            "or set the DISSERTATION_DATA_DIR environment variable."
        )
    MACHINE = 'Local'

OUTPUT_DIR = DATA_DIR.parent / 'outputs'
PROCESSED_DIR = OUTPUT_DIR / 'processed'
MODELS_DIR = OUTPUT_DIR / 'models'
RESULTS_DIR = OUTPUT_DIR / 'results'
FIGURES_DIR = OUTPUT_DIR / 'figures'
for d in [RESULTS_DIR, FIGURES_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Machine:    {MACHINE}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"pandas {pd.__version__} | SHAP {shap.__version__}")

## Load models and test data

In [ ]:
needed = [PROCESSED_DIR / 'test_predictions.parquet', MODELS_DIR / 'lightgbm.joblib',
          MODELS_DIR / 'logistic_regression_m0-4.joblib', MODELS_DIR / 'thresholds.json']
missing = [p.name for p in needed if not p.exists()]
if missing:
    raise FileNotFoundError(f"Missing {missing}. Run 2_Modelling.ipynb first.")

X_test = pd.read_parquet(PROCESSED_DIR / 'X_test.parquet')
preds = pd.read_parquet(PROCESSED_DIR / 'test_predictions.parquet')
assert X_test.index.equals(preds.index), "Test data and predictions are not aligned"

config = joblib.load(MODELS_DIR / 'feature_config.joblib')
FEATURES = config['FEATURES']
CAT_FEATURES = config['CAT_FEATURES']
NUM_FEATURES = config['NUM_FEATURES']
PROTECTED_ATTRIBUTES = config['PROTECTED_ATTRIBUTES']

lgbm_model = joblib.load(MODELS_DIR / 'lightgbm.joblib')
lr_model = joblib.load(MODELS_DIR / 'logistic_regression_m0-4.joblib')
with open(MODELS_DIR / 'thresholds.json') as f:
    thresholds = json.load(f)

# Background sample for the logistic regression explanation (the "average applicant")
X_train = pd.read_parquet(PROCESSED_DIR / 'X_train.parquet')
X_background = X_train.sample(min(50_000, len(X_train)), random_state=RANDOM_STATE)
del X_train
gc.collect()

y_test = preds['fraud_bool'].to_numpy()
lgbm_score = preds['lgbm_score'].to_numpy()
lgbm_flag = preds['lgbm_flag'].to_numpy()

# Columns where a negative value means "missing" (BAF datasheet)
MISSING_IF_NEGATIVE = ['prev_address_months_count', 'current_address_months_count',
                       'intended_balcon_amount', 'bank_months_count',
                       'session_length_in_minutes', 'device_distinct_emails_8w']

print("Test applications:", len(X_test))
print("LightGBM threshold:", round(thresholds['LightGBM'], 4))

## LightGBM: SHAP values for every test application

In [ ]:
start = time.time()
contrib = lgbm_model.predict(X_test, pred_contrib=True)   # exact TreeSHAP, last column = base value
print(f"TreeSHAP for {len(X_test):,} applications in {time.time() - start:.1f} s")

lgbm_shap = pd.DataFrame(contrib[:, :-1], columns=FEATURES, index=X_test.index).astype('float32')
lgbm_base = float(contrib[0, -1])
del contrib
gc.collect()

# Check: base value + SHAP values = the model's raw score (log-odds) for every application
raw_score = lgbm_model.predict(X_test, raw_score=True)
max_error = np.abs(lgbm_shap.sum(axis=1).to_numpy() + lgbm_base - raw_score).max()
print(f"Additivity check — largest difference: {max_error:.2e} (should be ~0)")
print(f"Base value (average log-odds): {lgbm_base:.3f} = probability {1 / (1 + np.exp(-lgbm_base)):.4f}")

# Numeric copy of the features for plotting (categories -> codes)
X_test_num = X_test.copy()
for col in CAT_FEATURES:
    X_test_num[col] = X_test_num[col].cat.codes
X_test_num = X_test_num.astype('float32')

## Global explanation: which features drive fraud scores?
**Mean |SHAP|** is the average size of a feature's effect on the score across all test applications, regardless of direction.

In [ ]:
importance = lgbm_shap.abs().mean().sort_values(ascending=False)
importance_pct = importance / importance.sum() * 100

top = importance_pct.head(20)[::-1]
colours = ['darkorange' if f in PROTECTED_ATTRIBUTES else 'tomato' for f in top.index]
plt.figure(figsize=(8, 7))
plt.barh(top.index, top.values, color=colours)
plt.xlabel('Share of total mean |SHAP| (%)')
plt.title('LightGBM: global feature importance (SHAP), top 20\n(orange = protected attribute)')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_global_importance.png', dpi=200, bbox_inches='tight')
plt.show()

print("Top 10 features:")
print(importance_pct.head(10).astype(float).round(1).to_string())

**Beeswarm plot** — each dot is one application. Position shows how much the feature pushed that application's score up (right) or down (left); colour shows whether the feature value was high (red) or low (blue). For categorical features (marked *) the colour has no meaning; their effects are shown separately below.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
plot_pos = rng.choice(len(X_test), size=min(N_PLOT, len(X_test)), replace=False)

plot_names = [f"{f} *" if f in CAT_FEATURES else f for f in FEATURES]
exp_sample = shap.Explanation(
    values=lgbm_shap.to_numpy()[plot_pos],
    base_values=np.full(len(plot_pos), lgbm_base),
    data=X_test_num.to_numpy()[plot_pos],
    feature_names=plot_names,
)

shap.plots.beeswarm(exp_sample, max_display=15, show=False)
plt.title('LightGBM: SHAP values for a random sample of test applications')
plt.savefig(FIGURES_DIR / 'shap_beeswarm.png', dpi=200, bbox_inches='tight')
plt.show()

## Categorical features: average effect of each category

In [ ]:
n = len(CAT_FEATURES)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, col in zip(axes, CAT_FEATURES):
    effect = lgbm_shap[col].groupby(X_test[col], observed=True).mean().sort_values()
    counts = X_test[col].value_counts()
    ax.barh(effect.index.astype(str), effect.values,
            color=['tomato' if v > 0 else 'steelblue' for v in effect.values])
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(col)
    ax.set_xlabel('Mean SHAP value (log-odds)')
    xmin, xmax = ax.get_xlim()
    ax.set_xlim(xmin, xmax + 0.35 * (xmax - xmin))   # room for the sample sizes
    ax.xaxis.set_major_locator(plt.MaxNLocator(4))
    for i, cat in enumerate(effect.index):   # sample sizes at the right-hand edge
        ax.text(0.99, i, f"n={counts[cat]:,}", transform=ax.get_yaxis_transform(),
                ha='right', va='center', fontsize=8, color='grey')
for ax in axes[n:]:
    ax.axis('off')

plt.suptitle('LightGBM: average SHAP value by category (positive = pushes towards fraud)', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_categorical_effects.png', dpi=200, bbox_inches='tight')
plt.show()

## Dependence plots: how the effect changes with the feature value
Top six non-categorical features. Grey points are **missing values** (encoded as negative numbers in BAF), which the model can treat as a separate signal.

In [ ]:
top_numeric = [f for f in importance.index if f not in CAT_FEATURES][:6]
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

for ax, col in zip(axes.ravel(), top_numeric):
    x = X_test_num[col].to_numpy()[plot_pos]
    y = lgbm_shap[col].to_numpy()[plot_pos]
    is_missing = (x < 0) if col in MISSING_IF_NEGATIVE else np.zeros_like(x, dtype=bool)
    ax.scatter(x[~is_missing], y[~is_missing], s=4, alpha=0.3, color='tomato')
    if is_missing.any():
        ax.scatter(x[is_missing], y[is_missing], s=4, alpha=0.3, color='grey', label='missing')
        ax.legend(loc='best', markerscale=3)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel(col)
    ax.set_ylabel('SHAP value (log-odds)')
    ax.set_title(col)

plt.suptitle('LightGBM: SHAP dependence plots for the top six numeric features', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_dependence.png', dpi=200, bbox_inches='tight')
plt.show()

## Local explanations: three individual decisions
One typical application from each outcome at the deployed LightGBM threshold (the case whose score is closest to the median score of its group):
- **True positive** — fraud, correctly flagged
- **False positive** — legitimate applicant, wrongly flagged (the Consumer Duty "foreseeable harm" case)
- **False negative** — fraud that was missed

The waterfall starts from the base value (average score) and shows how each feature moved this application's score.

In [ ]:
def typical_case(mask):
    """Position of the case whose score is closest to the median score in the group."""
    positions = np.flatnonzero(mask)
    scores = lgbm_score[positions]
    return int(positions[np.argmin(np.abs(scores - np.median(scores)))])

cases = {
    'true_positive': typical_case((y_test == 1) & (lgbm_flag == 1)),
    'false_positive': typical_case((y_test == 0) & (lgbm_flag == 1)),
    'false_negative': typical_case((y_test == 1) & (lgbm_flag == 0)),
}

summary = pd.DataFrame({
    name: {
        'row_id': X_test.index[pos],
        'actual': 'fraud' if y_test[pos] == 1 else 'legitimate',
        'flagged': bool(lgbm_flag[pos]),
        'fraud_score': round(float(lgbm_score[pos]), 4),
        'threshold': round(thresholds['LightGBM'], 4),
        'customer_age': X_test.iloc[pos]['customer_age'],
        'income': X_test.iloc[pos]['income'],
        'employment_status': X_test.iloc[pos]['employment_status'],
    }
    for name, pos in cases.items()
}).T
summary

In [ ]:
def waterfall(pos, title, filename, max_display=12):
    row = shap.Explanation(
        values=lgbm_shap.to_numpy()[pos],
        base_values=lgbm_base,
        data=X_test_num.to_numpy()[pos],
        display_data=X_test.iloc[pos].astype(object).to_numpy(),   # shows real category names
        feature_names=FEATURES,
    )
    shap.plots.waterfall(row, max_display=max_display, show=False)
    plt.title(f"{title}\nfraud score = {lgbm_score[pos]:.3f} | threshold = {thresholds['LightGBM']:.3f}")
    plt.savefig(FIGURES_DIR / filename, dpi=200, bbox_inches='tight')
    plt.show()

waterfall(cases['true_positive'], 'True positive: fraud, correctly flagged', 'shap_waterfall_true_positive.png')

In [ ]:
waterfall(cases['false_positive'], 'False positive: legitimate applicant, wrongly flagged', 'shap_waterfall_false_positive.png')

In [ ]:
waterfall(cases['false_negative'], 'False negative: fraud, missed', 'shap_waterfall_false_negative.png')

### From SHAP values to "reasons"
Consumer Duty's *consumer understanding* outcome implies that a firm should be able to say why a decision was made. The function below turns SHAP values into simple reason statements, like the reason codes used in credit decisions. Note how technical many features are (e.g. `name_email_similarity`, `velocity_6h`) — translating them into language a customer understands is a gap SHAP alone does not close, and is worth discussing in the dissertation.

In [ ]:
def fmt(v):
    # Readable value: whole numbers as integers, large numbers without decimals, small ones to 3 s.f.
    if isinstance(v, (float, np.floating)):
        if float(v).is_integer():
            return str(int(v))
        return f"{v:,.0f}" if abs(v) >= 100 else f"{v:.3g}"
    return str(v)

def reasons(pos, k=3):
    row_shap = lgbm_shap.iloc[pos]
    row_vals = X_test.iloc[pos]
    up = row_shap[row_shap > 0].sort_values(ascending=False).head(k)
    down = row_shap[row_shap < 0].sort_values().head(k)
    lines = [f"Fraud score {lgbm_score[pos]:.3f} (threshold {thresholds['LightGBM']:.3f}) -> "
             f"{'FLAGGED' if lgbm_flag[pos] else 'not flagged'}"]
    lines.append("  Main reasons the score went UP:")
    lines += [f"    - {f} = {fmt(row_vals[f])}  (+{v:.2f})" for f, v in up.items()]
    lines.append("  Main reasons the score went DOWN:")
    lines += [f"    - {f} = {fmt(row_vals[f])}  ({v:.2f})" for f, v in down.items()]
    return "\n".join(lines)

local_rows = []
for name, pos in cases.items():
    print(f"=== {name.replace('_', ' ').upper()} ===")
    print(reasons(pos))
    print()
    top5 = lgbm_shap.iloc[pos].abs().sort_values(ascending=False).head(5).index
    for rank, f in enumerate(top5, start=1):
        local_rows.append({'case': name, 'rank': rank, 'feature': f,
                           'value': X_test.iloc[pos][f], 'shap': float(lgbm_shap.iloc[pos][f])})

pd.DataFrame(local_rows).to_csv(RESULTS_DIR / 'shap_local_cases.csv', index=False)

## Cross-check against logistic regression
As set out in the proposal, LightGBM's explanations are sanity-checked against the interpretable baseline. Logistic regression SHAP values are computed with the exact linear formula, using 50,000 training applications as the "average applicant". One-hot columns are added back together, so each categorical feature gets one value, just as in LightGBM.

If both models rely on broadly the same features, that supports the SHAP explanation being meaningful rather than an artefact of the complex model.

In [ ]:
preprocess = lr_model.named_steps['preprocess']
classifier = lr_model.named_steps['classifier']

def to_dense(a):
    return a.toarray() if hasattr(a, 'toarray') else np.asarray(a)

Xt = to_dense(preprocess.transform(X_test))
background_mean = to_dense(preprocess.transform(X_background)).mean(axis=0)
coef = classifier.coef_[0]

# Map each transformed column back to its original feature
ohe = preprocess.named_transformers_['cat']
column_feature = list(NUM_FEATURES) + [col for col, cats in zip(CAT_FEATURES, ohe.categories_) for _ in cats]

lr_shap_cols = (Xt - background_mean) * coef
lr_base = float(classifier.intercept_[0] + background_mean @ coef)
lr_shap = (pd.DataFrame(lr_shap_cols, index=X_test.index)
           .T.groupby(np.array(column_feature)).sum().T[FEATURES].astype('float32'))
del Xt, lr_shap_cols
gc.collect()

max_error = np.abs(lr_shap.sum(axis=1).to_numpy() + lr_base - lr_model.decision_function(X_test)).max()
print(f"Additivity check — largest difference: {max_error:.2e} (should be ~0)")

In [ ]:
lr_importance = lr_shap.abs().mean()
comparison = pd.DataFrame({
    'LightGBM_share_%': importance_pct,
    'LogReg_share_%': lr_importance / lr_importance.sum() * 100,
})
comparison['LightGBM_rank'] = comparison['LightGBM_share_%'].rank(ascending=False).astype(int)
comparison['LogReg_rank'] = comparison['LogReg_share_%'].rank(ascending=False).astype(int)
comparison = comparison.sort_values('LightGBM_rank')
comparison.to_csv(RESULTS_DIR / 'shap_importance_comparison.csv')

rho, p_value = spearmanr(comparison['LightGBM_share_%'], comparison['LogReg_share_%'])
top10_overlap = len(set(comparison.index[comparison['LightGBM_rank'] <= 10]) &
                    set(comparison.index[comparison['LogReg_rank'] <= 10]))
print(f"Rank correlation of feature importance (Spearman): rho = {rho:.2f} (p = {p_value:.3g})")
print(f"Features in both models' top 10: {top10_overlap} of 10")

top15 = comparison.head(15)[::-1]
y = np.arange(len(top15))
plt.figure(figsize=(9, 7))
plt.barh(y + 0.2, top15['LightGBM_share_%'], height=0.4, color='tomato', label='LightGBM')
plt.barh(y - 0.2, top15['LogReg_share_%'], height=0.4, color='steelblue', label='Logistic regression')
plt.yticks(y, top15.index)
plt.xlabel('Share of total mean |SHAP| (%)')
plt.title(f"Feature importance: LightGBM vs logistic regression\n(Spearman rho = {rho:.2f}; top-10 overlap = {top10_overlap}/10)")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_lr_vs_lgbm.png', dpi=200, bbox_inches='tight')
plt.show()

comparison.head(15).round(2)

**Direction check.** For each numeric feature: does a higher value push the score the same way in both models? Logistic regression's direction is the sign of its coefficient; LightGBM's is the rank correlation between the feature value and its SHAP value (missing values excluded). Disagreements often point to non-linear effects that logistic regression cannot capture.

In [ ]:
num_coef = pd.Series(coef[:len(NUM_FEATURES)], index=NUM_FEATURES)

rows = []
for col in NUM_FEATURES:
    x = X_test[col].to_numpy(dtype=float)
    keep = (x >= 0) if col in MISSING_IF_NEGATIVE else np.ones_like(x, dtype=bool)
    s_vals = lgbm_shap[col].to_numpy()[keep]
    usable = keep.sum() > 2 and np.std(x[keep]) > 0 and np.std(s_vals) > 0   # constant -> no trend
    trend = spearmanr(x[keep], s_vals)[0] if usable else np.nan
    lr_dir = 'up' if num_coef[col] > 0 else 'down'
    lgbm_dir = 'unclear' if (np.isnan(trend) or abs(trend) < 0.1) else ('up' if trend > 0 else 'down')
    rows.append({'feature': col, 'LogReg_coefficient': num_coef[col], 'LogReg_direction': lr_dir,
                 'LightGBM_trend_rho': trend, 'LightGBM_direction': lgbm_dir,
                 'agree': 'n/a' if lgbm_dir == 'unclear' else ('yes' if lr_dir == lgbm_dir else 'NO')})

direction = (pd.DataFrame(rows).set_index('feature')
             .loc[[f for f in comparison.index if f in NUM_FEATURES]])
direction.to_csv(RESULTS_DIR / 'shap_direction_agreement.csv')
print(direction['agree'].value_counts().to_string())
direction.round(3)

## Protected attributes
How much do age, income and employment status contribute to each model's decisions — and does that contribution fall differently on different groups? The second part looks only at **legitimate applicants**, because a positive contribution for them can only increase the chance of being wrongly flagged. This leads directly into the fairness audit in notebook 4.

`customer_age` in BAF is rounded to the decade (10 = teens, 20 = twenties, …).

In [ ]:
protected_summary = comparison.loc[PROTECTED_ATTRIBUTES,
                                   ['LightGBM_share_%', 'LightGBM_rank', 'LogReg_share_%', 'LogReg_rank']]
protected_summary.loc['TOTAL (all three)'] = [
    protected_summary['LightGBM_share_%'].sum(), np.nan,
    protected_summary['LogReg_share_%'].sum(), np.nan,
]
protected_summary.to_csv(RESULTS_DIR / 'shap_protected_attributes.csv')
protected_summary.round(2)

In [ ]:
legit = y_test == 0
group_labels = {
    'customer_age': lambda s: s.map(lambda v: f"{int(v)}s"),
    'income': lambda s: s.map(lambda v: f"{v:.1f}"),
    'employment_status': lambda s: s.astype(str),
}

fig, axes = plt.subplots(1, len(PROTECTED_ATTRIBUTES), figsize=(6 * len(PROTECTED_ATTRIBUTES), 5))
for ax, col in zip(np.atleast_1d(axes), PROTECTED_ATTRIBUTES):
    groups = group_labels.get(col, lambda s: s.astype(str))(X_test.loc[legit, col])
    effect = pd.DataFrame({
        'LightGBM': lgbm_shap.loc[legit, col].groupby(groups.to_numpy()).mean(),
        'Logistic regression': lr_shap.loc[legit, col].groupby(groups.to_numpy()).mean(),
    })
    effect.plot(kind='bar', ax=ax, color=['tomato', 'steelblue'], edgecolor='black', width=0.8)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(f"{col} (legitimate applicants only)")
    ax.set_xlabel(col)
    ax.set_ylabel('Mean SHAP value of this feature (log-odds)')
    ax.tick_params(axis='x', rotation=0)

plt.suptitle('Contribution of each protected attribute to fraud scores, by group\n'
             '(positive = this attribute pushes legitimate applicants towards being flagged)', y=1.03)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'shap_protected_by_group.png', dpi=200, bbox_inches='tight')
plt.show()

## Save outputs
SHAP values for every test application are saved for notebook 4, so the fairness audit can link any group differences back to the features causing them.

**Caveats to discuss in the dissertation**
- TreeSHAP's default ("path-dependent") method can spread credit unevenly between **correlated features**, so a feature's importance may partly reflect a correlated partner (Slack et al., 2020 also show post-hoc explanations can be manipulated).
- SHAP explains **the model, not the world**: a large SHAP value for age means the model uses age, not that age causes fraud.
- BAF is synthetic, so the specific features driving fraud may differ from those in a real firm's data — the IEEE-CIS validation addresses this.

In [ ]:
lgbm_shap_out = lgbm_shap.copy()
lgbm_shap_out['base_value'] = np.float32(lgbm_base)
lgbm_shap_out.to_parquet(PROCESSED_DIR / 'shap_lgbm_test.parquet')

lr_shap_out = lr_shap.copy()
lr_shap_out['base_value'] = np.float32(lr_base)
lr_shap_out.to_parquet(PROCESSED_DIR / 'shap_logreg_test.parquet')

print("Saved:")
for p in sorted(PROCESSED_DIR.glob('shap_*.parquet')) + sorted(RESULTS_DIR.glob('shap_*.csv')) \
         + sorted(FIGURES_DIR.glob('shap_*.png')):
    print("  ", p.relative_to(OUTPUT_DIR.parent))